# Validação de Pneus — ApexSim (M1)

Este notebook valida as funções puras de `physics.py` contra:
1. **As curvas reais extraídas pelo Octávio da telemetria FastF1 / Kaggle 2023-2024 (entrega D5)**:
   - 39.966 voltas limpas em 44 GPs.
   - Parâmetros calibrados ($k_{1,quad}$ e $k_{2,lin}$) e pontos observados para SOFT, MEDIUM e HARD.
2. **Os critérios de aceite do roadmap (M1)**:
   - Funções puras sem acoplamento ao motor de simulação.
   - Macio degrada mais rápido que Médio, que degrada mais rápido que Duro.
   - Carro cheio é mais lento que carro vazio, na ordem de ~0,3 s por 10 kg.

Fundamentação matemática completa: `docs/fisica.md`, Seções 1 e 2.

In [ ]:
%matplotlib inline
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from physics import tyre_grip, tyre_time_loss, fuel_time_delta, TYRE_TABLE

data_path = Path("../../data/curvas_degradacao/curvas_degradacao.json")

with open(data_path, encoding="utf-8") as f:
    kaggle_data = json.load(f)

print(f"Fonte dos dados reais: {kaggle_data['meta']['fonte']}")
print(f"Total de voltas limpas analisadas: {kaggle_data['meta']['n_voltas_limpas']}")
compounds = ["soft", "medium", "hard"]


## 1. Curva de Degradação e Grip por Composto: Modelo vs. Dados Reais (FastF1/Kaggle D5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
colors = {"soft": "tab:red", "medium": "tab:orange", "hard": "tab:gray"}
comp_keys = {"soft": "SOFT", "medium": "MEDIUM", "hard": "HARD"}

ages_model = np.arange(0, 41)

for idx, c in enumerate(compounds):
    c_key = comp_keys[c]
    curva_real = kaggle_data["compostos"][c_key]["curva"]
    idades_reais = [pt["idade"] for pt in curva_real]
    perdas_reais = [pt["perda_s"] for pt in curva_real]
    erros_reais = [pt["erro_padrao_s"] for pt in curva_real]
    grip_real = [1.0 / (1.0 + max(0.0, pt["perda_s"]) / 3.0) for pt in curva_real]

    perdas_modelo = [tyre_time_loss(c, a) for a in ages_model]
    grip_modelo = [tyre_grip(c, a) for a in ages_model]

    ax_loss = axes[0, idx]
    ax_loss.errorbar(
        idades_reais,
        perdas_reais,
        yerr=erros_reais,
        fmt="o",
        color="black",
        alpha=0.6,
        markersize=4,
        capsize=2,
        label="Medido (FastF1 Kaggle)",
    )
    ax_loss.plot(
        ages_model,
        perdas_modelo,
        "-",
        color=colors[c],
        lw=2,
        label="Modelo (physics.py)",
    )
    ax_loss.set_title(f"{c.capitalize()} — Perda de Tempo (s)")
    ax_loss.set_ylabel("Perda de tempo (s)")
    ax_loss.grid(alpha=0.3)
    ax_loss.legend(fontsize=9)

    ax_grip = axes[1, idx]
    ax_grip.plot(
        idades_reais,
        grip_real,
        "o",
        color="black",
        alpha=0.6,
        markersize=4,
        label="Medido (FastF1 Kaggle)",
    )
    ax_grip.plot(
        ages_model,
        grip_modelo,
        "-",
        color=colors[c],
        lw=2,
        label="Modelo (physics.py)",
    )
    ax_grip.set_title(f"{c.capitalize()} — Grip Normalizado")
    ax_grip.set_xlabel("Idade do pneu (voltas)")
    ax_grip.set_ylabel("Grip normalizado (0 a 1)")
    ax_grip.grid(alpha=0.3)
    ax_grip.legend(fontsize=9)

plt.suptitle(
    "Validação do Modelo de Pneus vs Dados Reais FastF1 2023-2024 (Kaggle D5)",
    fontsize=13,
)
plt.tight_layout()
plt.savefig("validacao_pneus_grip.png", dpi=130)
plt.show()


## 2. Ordenação de degradação entre compostos

In [ ]:
age_check = 25
losses = {c: tyre_time_loss(c, age_check) for c in compounds}
print(f"Perda de tempo na volta {age_check} (idade do pneu):")
for c, v in losses.items():
    print(f"  {c:>6}: {v:.3f} s")

assert losses["soft"] > losses["medium"] > losses["hard"], "Ordem de degradação incorreta!"
print("\nOK: Soft > Medium > Hard em perda de tempo (macio degrada mais rápido).")

## 3. Penalidade de combustível (~0,3 s por 10 kg)

In [ ]:
fuel_levels = np.arange(0, 111, 10)
deltas = [fuel_time_delta(m) for m in fuel_levels]

plt.figure(figsize=(6, 4))
plt.plot(fuel_levels, deltas, marker="o", color="tab:blue")
plt.xlabel("Combustível a bordo (kg)")
plt.ylabel("Delta de tempo de volta (s)")
plt.title("Penalidade de tempo por massa de combustível")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("validacao_combustivel.png", dpi=130)
plt.show()

delta_10kg = fuel_time_delta(10) - fuel_time_delta(0)
print(f"Delta por 10kg: {delta_10kg:.3f} s (esperado ~0.3s, faixa aceita 0.2-0.5s)")
assert 0.2 <= delta_10kg <= 0.5
print("OK: penalidade de combustível na ordem de grandeza esperada.")


## 4. Resumo dos critérios "Pronto quando" (M1)

- [x] Funções puras, sem dependência do motor de eventos (`tyre_grip`, `fuel_time_delta` em `physics.py`).
- [x] Testes automatizados em `test_physics.py` (11 casos, todos passando).
- [x] Este notebook plota as curvas geradas vs. as curvas reais do Octávio (`data/curvas_degradacao/curvas_degradacao.json`).
- [x] Macio degrada mais rápido que Médio, que degrada mais rápido que Duro.
- [x] Carro cheio é mais lento que carro vazio, na ordem de ~0,3 s por 10 kg.

### Conclusão e Calibração M1
Os parâmetros $k_{1,quad}$ e $k_{2,lin}$ foram recalibrados com base nos 44 GPs de 2023-2024 (entrega D5), corrigindo a superestimação anterior de 2x a 3x em pneus macios e médios. Os parâmetros de *cliff* permanecem declarados como modelo teórico para idades extremas não observadas na telemetria.